In [1]:
import sys
import os

# Aggiunge la cartella radice del progetto al percorso di ricerca di Python
sys.path.append(os.path.abspath(os.path.join('..')))

In [4]:
import sentence_transformers
print(sentence_transformers.__version__)

model = SentenceTransformer()
print(model)           # se è vuoto, il caricamento è fallito
print(list(model.modules()))  # deve avere almeno Transformer + Pooling

3.2.1
SentenceTransformer(
  (0): None
)
[SentenceTransformer(
  (0): None
)]


In [ ]:
# Cella 1 - pulisci i file .pyc corrotti e reinstalla
import subprocess
subprocess.run(["find", "/opt/conda", "-name", "*.pyc", "-delete"])
subprocess.run(["pip", "install", "--force-reinstall", "--no-cache-dir", 
                "torch", "sentence-transformers"])

In [ ]:
import app.core.config as config
#import app.core.json_prompt_read as json_prompt_read
from sentence_transformers import SentenceTransformer


class PromptEmbedder:
    def __init__(self, model):
        self.model = model

    def embed(self, prompt):
        # Use the model to generate an embedding for the prompt
        embedding = self.model.encode(prompt)
        return embedding

if __name__ == "__main__":

    # Load the model specified in the config
    model_name = config.EMBEDDING_MODEL
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

    embedder = PromptEmbedder(model)
    #json_path = "prompt.json"

    #prompt = json_prompt_read.read_json_prompt(json_path)
    prompt = "ricetta con pasta, zucchine e tonno"

    # Get the embedding for the prompt
    query_embedding = embedder.embed(prompt)
  
    print(query_embedding)

In [8]:
import chromadb
client = chromadb.HttpClient(host="chroma", port=8000)
print([c.name for c in client.list_collections()])

[]


In [6]:
###CONFRONTA CON I DATI NEL DB
import chromadb
from chromadb.config import Settings

client = chromadb.HttpClient(
    host="chroma",        # nome del servizio nel compose
    port=8000
)

collection = client.get_collection("ricette")
print(collection.count())

5652


In [11]:
# L'embedding del prompt è già calcolato nella cella precedente
results = collection.query(
    query_embeddings=[query_embedding.tolist()],  # converti numpy array in lista
    n_results=10,                           # top 10 più simili
    include=["documents", "distances", "metadatas"]
)

# Stampa i risultati
print("Query dell'utente: " + prompt)
for i, (doc, distance) in enumerate(zip(results["documents"][0], results["distances"][0])):
    similarity = 1 - distance  # ChromaDB restituisce la distanza, non la similarità
    print(f"\n--- Risultato {i+1} (similarità: {similarity:.4f}) ---")
    print(doc)

Query dell'utente: ricetta con pasta, zucchine e tonno

--- Risultato 1 (similarità: 0.4054) ---
Titolo: Spaghetti con tonno e zucchine
Categoria: Primi piatti
Difficoltà: Molto facile
Tempo di preparazione: 15 min
Tempo di cottura: 10 min
Costo: Molto basso
Dosi: 4 persone
Ingredienti: Spaghetti (320 g), Zucchine (350 g), Tonno sott'olio (320 g), Olio extravergine d'oliva (20 g), Aglio (1 spicchio), Timo (2 rametti), Olio di semi di arachide (per friggere le zucchine 300 g), Sale fino (q.b.), Pepe nero (q.b.)
Tag dietetici: Senza lattosio
Valori nutrizionali: Energia: 854,5 Kcal, Proteine: 35 g, Carboidrati: 63,7 g, Grassi: 51 g

--- Risultato 2 (similarità: 0.3838) ---
Titolo: Pasta zucchine e zafferano
Categoria: BENESSERE
Difficoltà: Facile
Tempo di preparazione: 10 min
Tempo di cottura: 20 min
Dosi: 4 persone
Ingredienti: Pipe Rigate (320 g), Zucchine (500 g), Zafferano ((2 bustine da 0,15 g) 0,3 g), Grana Padano DOP (30 g), Menta (q.b.), Olio extravergine d'oliva (q.b.), Sale fin

In [ ]:
client = chromadb.HttpClient(host="localhost", port=8000)